**VAE run**

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('../minmax.xlsx')
data = df.values
labels = pd.read_csv('../idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Variational Autoencoder-Classifier Model

class JointVAEClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointVAEClassifier, self).__init__()
        # Encoder
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128)
        )
        # Layers to produce the mean and log-variance for the latent distribution
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode input to get intermediate representation
        x_encoded = self.encoder_net(x)
        # Produce latent distribution parameters
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        # Sample latent vector
        z = self.reparameterize(mu, logvar)
        # Reconstruct input and classify based on latent vector
        reconstruction = self.decoder(z)
        logits = self.classifier(z)
        return reconstruction, logits, mu, logvar


# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance reconstruction and KL loss and classification loss
lambda_recon = 0.5

def vae_combined_loss(reconstructed, original, logits, labels, mu, logvar):
    # Reconstruction loss
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    # KL divergence loss: average over batch
    loss_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # Classification loss
    loss_class = classification_loss_fn(logits, labels)
    # Combined loss: balance reconstruction (with KL) and classification
    return lambda_recon * (loss_recon + loss_kl) + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointVAEClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50


# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits, mu, logvar = model(inputs)
        loss = vae_combined_loss(reconstruction, inputs, logits, labels, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_encoded = model.encoder_net(X_train)
    train_latent = model.fc_mu(train_encoded).numpy()
    
    test_encoded = model.encoder_net(X_test)
    test_latent = model.fc_mu(test_encoded).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

# Import necessary libraries
import numpy as np
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
import warnings
warnings.filterwarnings("ignore")

# Define the fitness function for BGWO
def fitness_function(position):
    # Select features based on the binary position vector
    selected_features = [i for i, bit in enumerate(position) if bit > 0.5]
    if not selected_features:  # Avoid empty feature subsets
        return 0.0
    
    model = GaussianNB()
    stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    X_sel = train_latent[:, selected_features]
    scores = cross_val_score(model, X_sel, y_train_np, cv=stratified_kfold, scoring='accuracy')
    return scores.mean()

# Initialize BGWO parameters
num_wolves = 10
num_features = latent_dim  # Number of features in the dataset
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly (binary positions for feature selection)
wolves = np.random.randint(0, 2, size=(num_wolves, num_features))

for i in range(num_wolves):
    num_selected = np.random.randint(1, num_features + 1)
    selected = np.random.choice(num_features, num_selected, replace=False)
    wolves[i, selected] = 1

# BGWO algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(wolf) for wolf in wolves])
    
    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]
    
    a = 2 - 2 * iteration / max_iterations
    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            # Calculate A and C for alpha, beta, delta
            A1 = 2 * a * np.random.rand() - a
            C1 = 2 * np.random.rand()
            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            X1 = alpha[j] - A1 * D_alpha
            
            A2 = 2 * a * np.random.rand() - a
            C2 = 2 * np.random.rand()
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            X2 = beta[j] - A2 * D_beta
            
            A3 = 2 * a * np.random.rand() - a
            C3 = 2 * np.random.rand()
            D_delta = abs(C3 * delta[j] - wolves[i, j])
            X3 = delta[j] - A3 * D_delta
            
            # Update position with sigmoid transfer
            new_pos = (X1 + X2 + X3) / 3
            prob = 1 / (1 + np.exp(-new_pos))  # Sigmoid
            wolves[i, j] = 1 if np.random.rand() < prob else 0
# Evaluate the best solution
best_wolf = alpha
selected_features = [i for i in range(len(best_wolf)) if best_wolf[i] > 0.5]

if not selected_features:
    print(" No features selected by BGWO. Using all features as fallback.")
    selected_features = list(range(num_features))  # fallback: use all features

print("Selected features: ", selected_features)

X_train_selected = train_latent[:, selected_features]
X_test_selected = test_latent[:, selected_features]


svm_model = SVC(random_state=42)
svm_model.fit(X_train_selected, y_train_np)
svm_preds = svm_model.predict(X_test_selected)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print("SVM")
print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

nbc_model = GaussianNB()
nbc_model.fit(X_train_selected, y_train_np)
nbc_preds = nbc_model.predict(X_test_selected)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print("NBC")
print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_selected, y_train_np)
rf_preds = rf_model.predict(X_test_selected)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print("RF")
print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.4878
Epoch [2/50], Loss: 1.2366
Epoch [3/50], Loss: 1.1418
Epoch [4/50], Loss: 1.0105
Epoch [5/50], Loss: 0.9459
Epoch [6/50], Loss: 0.9283
Epoch [7/50], Loss: 0.8376
Epoch [8/50], Loss: 0.8214
Epoch [9/50], Loss: 0.7925
Epoch [10/50], Loss: 0.7590
Epoch [11/50], Loss: 0.7423
Epoch [12/50], Loss: 0.7994
Epoch [13/50], Loss: 0.7074
Epoch [14/50], Loss: 0.7493
Epoch [15/50], Loss: 0.7386
Epoch [16/50], Loss: 0.7331
Epoch [17/50], Loss: 0.7920
Epoch [18/50], Loss: 0.6983
Epoch [19/50], Loss: 0.7322
Epoch [20/50], Loss: 0.7147
Epoch [21/50], Loss: 0.6961
Epoch [22/50], Loss: 0.7935
Epoch [23/50], Loss: 0.7111
Epoch [24/50], Loss: 0.6835
Epoch [25/50], Loss: 0.7118
Epoch [26/50], Loss: 0.6873
Epoch [27/50], Loss: 0.7355
Epoch [28/50], Loss: 0.6357
Epoch [29/50], Loss: 0.7441
Epoch [30/50], Loss: 0.6544
Epoch [31/50], Loss: 0.6662
Epoch [32/50], Loss: 0.6257
Epoch [33/50], Loss: 0.6868
Epoch [34/50], Loss: 0.6876
Epoch [35/50], Loss: 0.7102


**BGWO with SVM training**

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('../minmax.xlsx')
data = df.values
labels = pd.read_csv('../idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Variational Autoencoder-Classifier Model

class JointVAEClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointVAEClassifier, self).__init__()
        # Encoder
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128)
        )
        # Layers to produce the mean and log-variance for the latent distribution
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode input to get intermediate representation
        x_encoded = self.encoder_net(x)
        # Produce latent distribution parameters
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        # Sample latent vector
        z = self.reparameterize(mu, logvar)
        # Reconstruct input and classify based on latent vector
        reconstruction = self.decoder(z)
        logits = self.classifier(z)
        return reconstruction, logits, mu, logvar


# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance reconstruction and KL loss and classification loss
lambda_recon = 0.5

def vae_combined_loss(reconstructed, original, logits, labels, mu, logvar):
    # Reconstruction loss
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    # KL divergence loss: average over batch
    loss_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # Classification loss
    loss_class = classification_loss_fn(logits, labels)
    # Combined loss: balance reconstruction (with KL) and classification
    return lambda_recon * (loss_recon + loss_kl) + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointVAEClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50


# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits, mu, logvar = model(inputs)
        loss = vae_combined_loss(reconstruction, inputs, logits, labels, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_encoded = model.encoder_net(X_train)
    train_latent = model.fc_mu(train_encoded).numpy()
    
    test_encoded = model.encoder_net(X_test)
    test_latent = model.fc_mu(test_encoded).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

# Import necessary libraries
import numpy as np
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
import warnings
warnings.filterwarnings("ignore")

# Define the fitness function for BGWO
def fitness_function(position):
    # Select features based on the binary position vector
    selected_features = [i for i, bit in enumerate(position) if bit > 0.5]
    if not selected_features:  # Avoid empty feature subsets
        return 0.0
    
    model = SVC(random_state=42)
    stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    X_sel = train_latent[:, selected_features]
    scores = cross_val_score(model, X_sel, y_train_np, cv=stratified_kfold, scoring='accuracy')
    return scores.mean()

# Initialize BGWO parameters
num_wolves = 10
num_features = latent_dim  # Number of features in the dataset
max_iterations = 20
alpha, beta, delta = None, None, None

# Initialize the wolves' positions randomly (binary positions for feature selection)
wolves = np.random.randint(0, 2, size=(num_wolves, num_features))

for i in range(num_wolves):
    num_selected = np.random.randint(1, num_features + 1)
    selected = np.random.choice(num_features, num_selected, replace=False)
    wolves[i, selected] = 1

# BGWO algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(wolf) for wolf in wolves])
    
    # Update alpha, beta, and delta wolves
    sorted_indices = np.argsort(fitness)[::-1]
    alpha, beta, delta = wolves[sorted_indices[:3]]
    
    a = 2 - 2 * iteration / max_iterations
    # Update positions of wolves
    for i in range(num_wolves):
        for j in range(num_features):
            # Calculate A and C for alpha, beta, delta
            A1 = 2 * a * np.random.rand() - a
            C1 = 2 * np.random.rand()
            D_alpha = abs(C1 * alpha[j] - wolves[i, j])
            X1 = alpha[j] - A1 * D_alpha
            
            A2 = 2 * a * np.random.rand() - a
            C2 = 2 * np.random.rand()
            D_beta = abs(C2 * beta[j] - wolves[i, j])
            X2 = beta[j] - A2 * D_beta
            
            A3 = 2 * a * np.random.rand() - a
            C3 = 2 * np.random.rand()
            D_delta = abs(C3 * delta[j] - wolves[i, j])
            X3 = delta[j] - A3 * D_delta
            
            # Update position with sigmoid transfer
            new_pos = (X1 + X2 + X3) / 3
            prob = 1 / (1 + np.exp(-new_pos))  # Sigmoid
            wolves[i, j] = 1 if np.random.rand() < prob else 0
# Evaluate the best solution
best_wolf = alpha
selected_features = [i for i in range(len(best_wolf)) if best_wolf[i] > 0.5]

if not selected_features:
    print(" No features selected by BGWO. Using all features as fallback.")
    selected_features = list(range(num_features))  # fallback: use all features

print("Selected features: ", selected_features)

X_train_selected = train_latent[:, selected_features]
X_test_selected = test_latent[:, selected_features]


svm_model = SVC(random_state=42)
svm_model.fit(X_train_selected, y_train_np)
svm_preds = svm_model.predict(X_test_selected)

svm_acc = accuracy_score(y_test_np, svm_preds)
svm_percision = precision_score(y_test_np, svm_preds, average='macro')
svm_recall = recall_score(y_test_np, svm_preds, average='macro')
svm_f1 = f1_score(y_test_np, svm_preds, average='macro')

print("SVM")
print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")
print(f"SVM Percision on latent features selected by GA: {svm_percision * 100:.4f}")
print(f"SVM Recall on latent features selected by GA: {svm_recall * 100:.4f}")
print(f"SVM F1-Score on latent features selected by GA: {svm_f1 * 100:.4f}")

nbc_model = GaussianNB()
nbc_model.fit(X_train_selected, y_train_np)
nbc_preds = nbc_model.predict(X_test_selected)

nbc_acc = accuracy_score(y_test_np, nbc_preds)
nbc_percision = precision_score(y_test_np, nbc_preds, average='macro')
nbc_recall = recall_score(y_test_np, nbc_preds, average='macro')
nbc_f1 = f1_score(y_test_np, nbc_preds, average='macro')

print("NBC")
print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")
print(f"Naïve Bayes Percision on latent features selected by GA: {nbc_percision * 100:.4f}")
print(f"Naïve Bayes Recall on latent features selected by GA: {nbc_recall * 100:.4f}")
print(f"Naïve Bayes F1-Score on latent features selected by GA: {nbc_f1 * 100:.4f}")

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_selected, y_train_np)
rf_preds = rf_model.predict(X_test_selected)

rf_acc = accuracy_score(y_test_np, rf_preds)
rf_percision = precision_score(y_test_np, rf_preds, average='macro')
rf_recall = recall_score(y_test_np, rf_preds, average='macro')
rf_f1 = f1_score(y_test_np, rf_preds, average='macro')

print("RF")
print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")
print(f"Random Forest Percision on latent features selected by GA: {rf_percision * 100:.4f}")
print(f"Random Forest Recall on latent features selected by GA: {rf_recall * 100:.4f}")
print(f"Random Forest F1-Score on latent features selected by GA: {rf_f1 * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.5085
Epoch [2/50], Loss: 1.2765
Epoch [3/50], Loss: 1.1209
Epoch [4/50], Loss: 1.0937
Epoch [5/50], Loss: 0.9343
Epoch [6/50], Loss: 0.8706
Epoch [7/50], Loss: 0.8300
Epoch [8/50], Loss: 0.8638
Epoch [9/50], Loss: 0.8369
Epoch [10/50], Loss: 0.7990
Epoch [11/50], Loss: 0.7688
Epoch [12/50], Loss: 0.7290
Epoch [13/50], Loss: 0.7540
Epoch [14/50], Loss: 0.7725
Epoch [15/50], Loss: 0.7279
Epoch [16/50], Loss: 0.6997
Epoch [17/50], Loss: 0.7834
Epoch [18/50], Loss: 0.7061
Epoch [19/50], Loss: 0.7644
Epoch [20/50], Loss: 0.7226
Epoch [21/50], Loss: 0.7018
Epoch [22/50], Loss: 0.6879
Epoch [23/50], Loss: 0.6848
Epoch [24/50], Loss: 0.6720
Epoch [25/50], Loss: 0.7021
Epoch [26/50], Loss: 0.6808
Epoch [27/50], Loss: 0.6693
Epoch [28/50], Loss: 0.6717
Epoch [29/50], Loss: 0.6549
Epoch [30/50], Loss: 0.6683
Epoch [31/50], Loss: 0.7096
Epoch [32/50], Loss: 0.6894
Epoch [33/50], Loss: 0.6656
Epoch [34/50], Loss: 0.6485
Epoch [35/50], Loss: 0.6315
